In [2]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, field_validator


class Entrada(BaseModel):
    # TODO
    model_config = ConfigDict(extra="forbid")
    
    id_paquete: str = Field(min_length=1)
    peso_kg: float = Field(gt=0,le=30)
    distancia_km: int = Field(ge=0,le=200)

    @field_validator("id_paquete", mode="before")
    @classmethod
    def quitar_espacios(cls, value):
        new_value = value.strip()
        return new_value
    pass


class Salida(BaseModel):
    # TODO
    model_config = ConfigDict(extra="forbid")

    id_paquete: str = Field(min_length=1)
    categoria: str = Field(Literal["normal","urgente"])
    confianza: float = Field(ge=0,le=1)
    
    pass

In [49]:
import csv

with open('C:/Users/Guille/Documents/Máster/Operación de Modelos/Examen_S1_S3_material_alumnos/clasificador-paquetes/data/raw/paquetes.csv') as csvfile:
    reader = csv.DictReader(csvfile)
    for rows in reader:
        print(rows)


{'id_paquete': ' P01 ', 'peso_kg': '2.36', 'distancia_km': '8'}
{'id_paquete': 'P02', 'peso_kg': '5.0', 'distancia_km': '75'}
{'id_paquete': 'P03', 'peso_kg': '18.44', 'distancia_km': '12'}
{'id_paquete': 'P04', 'peso_kg': '7.04', 'distancia_km': '55'}
{'id_paquete': 'P05', 'peso_kg': '1.26', 'distancia_km': '20'}
{'id_paquete': 'P06', 'peso_kg': '12.55', 'distancia_km': '120'}


In [38]:
prueba_paquete = {'id_paquete': ' P01 ', 'peso_kg': '2.36', 'distancia_km': '8'}

In [40]:
nueva_fila = Entrada(**prueba_paquete)

In [53]:
from pathlib import Path
def leer_csv(ruta: Path) -> list[Entrada]:
    # TODO
    with open(ruta, newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        paquetes = list()
        for rows in reader:
            try:
                new_entrada = Entrada(**rows)
                paquetes.append(new_entrada)
            except:
                    raise ValueError
    return paquetes
    raise NotImplementedError

In [51]:
leer_csv('C:/Users/Guille/Documents/Máster/Operación de Modelos/Examen_S1_S3_material_alumnos/clasificador-paquetes/data/raw/paquetes.csv')

NotImplementedError: 

In [52]:
with open('C:/Users/Guille/Documents/Máster/Operación de Modelos/Examen_S1_S3_material_alumnos/clasificador-paquetes/data/raw/paquetes.csv') as csvfile:
    reader = csv.DictReader(csvfile)
    paquetes = list()
    for rows in reader:
        new_entrada = Entrada(**rows)
        paquetes.append(new_entrada)

In [57]:
paquetes[0].id_paquete

'P01'

In [86]:
prueba_paquete = {'id_paquete': ' P01 ', 'peso_kg': '2.36', 'distancia_km': '8'}
prueba_paquete

{'id_paquete': ' P01 ', 'peso_kg': '2.36', 'distancia_km': '8'}

In [88]:
nueva_fila = Entrada(**prueba_paquete)
nueva_fila

Entrada(id_paquete='P01', peso_kg=2.36, distancia_km=8)

In [66]:
resultado = list()
peso_redondeado = round(nueva_fila.peso_kg, 1)
resultado.append([peso_redondeado,nueva_fila.distancia_km])

In [67]:
resultado

[[2.4, 8]]

In [69]:
import joblib
model = joblib.load('C:/Users/Guille/Documents/Máster/Operación de Modelos/Examen_S1_S3_material_alumnos/clasificador-paquetes/models/modelo.joblib')

c:\Users\Guille\Documents\Máster\Operación de Modelos\Examen_S1_S3_material_alumnos\clasificador-paquetes\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [72]:
model.predict(resultado)

array(['normal'], dtype='<U7')

In [81]:
model.predict_proba(resultado)

array([[1., 0.]])

In [79]:
model.predict([[6.0, 100]])

array(['urgente'], dtype='<U7')

In [80]:
model.predict_proba([[6.0, 100]])

array([[0., 1.]])

In [84]:
a = model.predict_proba([[6.0, 100]])
a[0][1]

np.float64(1.0)

In [89]:
def preprocesar(entrada: Entrada) -> list[float]:
    # TODO
    resultado = list()
    peso_redondeado = round(entrada.peso_kg, 1)
    resultado.append([peso_redondeado,entrada.distancia_km])

    return resultado
    raise NotImplementedError


def cargar_modelo(ruta: Path):
    # TODO
    modelo = joblib.load(ruta)
    return modelo
    raise NotImplementedError


def predecir(entrada: Entrada, modelo) -> Salida:
    # TODO
    valores = preprocesar(entrada)
    id_paquete = entrada.id_paquete
    categoria = modelo.predict(valores)
    if categoria == "urgente":
        confianza = modelo.predict_proba(valores)[0][1]
    else:
        confianza = modelo.predict_proba(valores)[0][0]

    dict_resultado ={'id_paquete': id_paquete, 'categoria': categoria, 'confianza': confianza}
    resultado = Salida(**dict_resultado)

    return resultado
    raise NotImplementedError

In [90]:
lista_prueba = list()
dict_resultado1 ={'id_paquete': 'P1', 'categoria': 'urgente', 'confianza': 1.0}
dict_resultado2 ={'id_paquete': 'P1', 'categoria': 'urgente', 'confianza': 1.0}
lista_prueba.append(Salida(**dict_resultado1))
lista_prueba.append(Salida(**dict_resultado2))


In [91]:
lista_prueba

[Salida(id_paquete='P1', categoria='urgente', confianza=1.0),
 Salida(id_paquete='P1', categoria='urgente', confianza=1.0)]

In [93]:
import pandas as pd

In [94]:
lista_prueba.to_dataframe()

AttributeError: 'list' object has no attribute 'to_dataframe'

In [98]:
df = pd.DataFrame(columns=['id_paquete','categoria','confianza'])

In [99]:
df

,id_paquete,categoria,confianza


In [103]:
for i in lista_prueba:
    print(i)
    df.loc[len(df)] = [i.id_paquete, i.categoria, i.confianza]

id_paquete='P1' categoria='urgente' confianza=1.0
id_paquete='P1' categoria='urgente' confianza=1.0


In [102]:
df

,id_paquete,categoria,confianza
0,P1,28,Madrid
1,P1,28,Madrid
